In [1]:
"""
lighting_app.py V0.1
---------------
Tkinter mini-app for placing lights in AutoCAD.
Run while AutoCAD is open with your drawing loaded.

Requirements:
    pip install pyautocad pywin32
    (tkinter is built into Python)
"""

import math
import sys
import array as array_mod
import threading
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import win32com.client
import pythoncom

def make_point(x, y, z=0.0):
    """Create a VARIANT 3D point compatible with raw AutoCAD COM."""
    pt = win32com.client.VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_R8, [x, y, z])
    return pt

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "Panneau LED 60×60":    "PANNEAU_LED_60x60",
    "Spot CoreLine DN140B": "SPOT_CORELINE_DN140B",
    "Hublot étanche 11W":   "HUBLOT_ETANCHE_11W",
    "Applique étanche 11W": "APPLIQUE_ETANCHE_11W",
    "Brasseur d'air 75W":   "BRASSEUR_AIR_75W",
}

LAYER_MAP = {
    "PANNEAU_LED_60x60":    "ECLAIRAGE-PANNEAU",
    "SPOT_CORELINE_DN140B": "ECLAIRAGE-SPOT",
    "HUBLOT_ETANCHE_11W":   "ECLAIRAGE-HUBLOT",
    "APPLIQUE_ETANCHE_11W": "ECLAIRAGE-APPLIQUE",
    "BRASSEUR_AIR_75W":     "ECLAIRAGE-VENTILATEUR",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"
MARGIN_RATIO = 0.001
BLOCK_SCALE  = 1.0

# ─────────────────────────────────────────────
# CORE LOGIC (same as CLI version)
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


def draw_bulge_wire(ms, pts, bulge):
    if len(pts) < 2:
        return None
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])
    bulge_vals = []
    for i in range(len(pts) - 1):
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))
    bulge_vals.append(0.0)

    # AutoCAD COM requires a VARIANT array of doubles
    pt_variant = win32com.client.VARIANT(
        pythoncom.VT_ARRAY | pythoncom.VT_R8, flat_pts)

    pline = ms.AddLightWeightPolyline(pt_variant)
    pline = win32com.client.Dispatch(pline)
    pline.Closed = False
    for i, b in enumerate(bulge_vals):
        pline.SetBulge(i, b)
    pline.Update()
    return pline


def add_label(ms, x, y, text, height, offset_y, offset_x=0.0):
    try:
        lbl = ms.AddMText(make_point(x + offset_x, y - offset_y, 0.0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        # Apply Times New Roman Bold using MText formatting codes
        # {\fTimes New Roman|b1|i0|c0|p0; ...text... } sets font+bold for the content
        lbl.TextString = "{\\fTimes New Roman|b1|i0;" + text + "}"
        return lbl
    except Exception:
        return None


# ─────────────────────────────────────────────
# TKINTER APP
# ─────────────────────────────────────────────

class LightingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("AutoCAD Lighting Placer")
        self.resizable(False, False)
        self.configure(bg="#1e1e2e")

        self.rectangles = []   # list of bounds tuples (plain Python, thread-safe)
        self._acad = None
        self._doc  = None
        self._ms   = None
        import queue
        self._job_queue = queue.Queue()

        self._build_ui()
        self._connect_autocad()

    # ── UI BUILD ────────────────────────────

    def _build_ui(self):
        PAD  = 12
        FONT = ("Segoe UI", 10)
        FONT_BOLD = ("Segoe UI", 10, "bold")
        BG   = "#1e1e2e"
        CARD = "#2a2a3e"
        ACC  = "#7c3aed"   # purple accent
        FG   = "#e2e8f0"
        ENTRY_BG = "#313145"

        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("TLabel",      background=CARD,  foreground=FG,  font=FONT)
        style.configure("TFrame",      background=CARD)
        style.configure("TLabelframe", background=CARD,  foreground=FG,  font=FONT_BOLD)
        style.configure("TLabelframe.Label", background=CARD, foreground=ACC, font=FONT_BOLD)
        style.configure("TCombobox",   fieldbackground=ENTRY_BG, background=ENTRY_BG,
                        foreground=FG, font=FONT)
        style.configure("TCheckbutton", background=CARD, foreground=FG, font=FONT)
        style.map("TCheckbutton", background=[("active", CARD)])

        outer = tk.Frame(self, bg=BG, padx=PAD, pady=PAD)
        outer.pack(fill="both", expand=True)

        # ── Header ──────────────────────────
        hdr = tk.Frame(outer, bg=ACC, pady=8)
        hdr.pack(fill="x", pady=(0, PAD))
        tk.Label(hdr, text="⚡  AutoCAD Lighting Placer",
                 font=("Segoe UI", 14, "bold"), bg=ACC, fg="white").pack()

        # ── Status bar ──────────────────────
        self.status_var = tk.StringVar(value="Connecting to AutoCAD…")
        status_bar = tk.Frame(outer, bg=CARD, padx=8, pady=5)
        status_bar.pack(fill="x", pady=(0, PAD))
        self.status_dot = tk.Label(status_bar, text="●", font=("Segoe UI", 12),
                                   bg=CARD, fg="#f59e0b")
        self.status_dot.pack(side="left")
        tk.Label(status_bar, textvariable=self.status_var,
                 font=FONT, bg=CARD, fg=FG).pack(side="left", padx=6)

        # ── Two columns ─────────────────────
        cols = tk.Frame(outer, bg=BG)
        cols.pack(fill="both")

        left  = tk.Frame(cols, bg=BG)
        right = tk.Frame(cols, bg=BG)
        left.pack(side="left", fill="both", padx=(0, 6))
        right.pack(side="left", fill="both")

        # ── LEFT: Room + Fixture ─────────────
        room_frame = ttk.LabelFrame(left, text="  Room Selection", padding=10)
        room_frame.pack(fill="x", pady=(0, 8))

        tk.Label(room_frame, text="Rectangles found:", bg=CARD, fg=FG,
                 font=FONT).grid(row=0, column=0, sticky="w", pady=2)

        self.rect_listbox = tk.Listbox(
            room_frame, height=5, selectmode="multiple",
            bg=ENTRY_BG, fg=FG, font=FONT,
            selectbackground=ACC, selectforeground="white",
            borderwidth=0, highlightthickness=1,
            highlightcolor=ACC, relief="flat"
        )
        self.rect_listbox.grid(row=1, column=0, columnspan=2, sticky="ew", pady=4)

        btn_row = tk.Frame(room_frame, bg=CARD)
        btn_row.grid(row=2, column=0, columnspan=2, sticky="ew")
        self._btn(btn_row, "⟳  Scan", self._scan_rectangles, ACC).pack(side="left", padx=(0,4))
        self._btn(btn_row, "Select All", self._select_all).pack(side="left", padx=(4,0))
        self._btn(btn_row, "⚡ Reconnect", self._connect_autocad, "#b45309").pack(side="left", padx=(4,0))

        fix_frame = ttk.LabelFrame(left, text="  Fixture", padding=10)
        fix_frame.pack(fill="x", pady=(0, 8))

        tk.Label(fix_frame, text="Type:", bg=CARD, fg=FG, font=FONT).grid(
            row=0, column=0, sticky="w", pady=2)
        self.fixture_var = tk.StringVar()
        fix_combo = ttk.Combobox(fix_frame, textvariable=self.fixture_var,
                                 values=list(BLOCK_NAMES.keys()),
                                 state="readonly", width=26)
        fix_combo.grid(row=0, column=1, sticky="ew", padx=(6,0))
        fix_combo.current(0)

        tk.Label(fix_frame, text="Count per room:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.count_var = tk.StringVar(value="6")
        self._entry(fix_frame, self.count_var).grid(row=1, column=1, sticky="ew", padx=(6,0))

        # ── RIGHT: Wire + Label ──────────────
        wire_frame = ttk.LabelFrame(right, text="  Wire", padding=10)
        wire_frame.pack(fill="x", pady=(0, 8))

        self.wire_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(wire_frame, text="Draw arc wire", variable=self.wire_var,
                        command=self._toggle_wire).grid(row=0, column=0, columnspan=2,
                                                        sticky="w", pady=2)

        tk.Label(wire_frame, text="Bulge:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.bulge_var = tk.DoubleVar(value=0.5)
        self.bulge_scale = tk.Scale(
            wire_frame, from_=-1.0, to=1.0, resolution=0.05,
            orient="horizontal", variable=self.bulge_var,
            bg=CARD, fg=FG, troughcolor=ENTRY_BG,
            highlightthickness=0, activebackground=ACC,
            length=160, font=("Segoe UI", 8)
        )
        self.bulge_scale.grid(row=1, column=1, sticky="ew", padx=(6,0))

        self.bulge_lbl = tk.Label(wire_frame, text="≈ 106° arc",
                                  bg=CARD, fg="#94a3b8", font=("Segoe UI", 9))
        self.bulge_lbl.grid(row=2, column=1, sticky="w", padx=(6,0))
        self.bulge_var.trace_add("write", self._update_bulge_label)

        lbl_frame = ttk.LabelFrame(right, text="  Circuit Labels", padding=10)
        lbl_frame.pack(fill="x", pady=(0, 8))

        self.label_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(lbl_frame, text="Add labels", variable=self.label_var,
                        command=self._toggle_labels).grid(row=0, column=0, columnspan=2,
                                                          sticky="w", pady=2)

        # Label mode: Fixed or Auto-increment
        tk.Label(lbl_frame, text="Mode:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.label_mode_var = tk.StringVar(value="auto")
        mode_frame = tk.Frame(lbl_frame, bg=CARD)
        mode_frame.grid(row=1, column=1, sticky="w", padx=(6,0))
        ttk.Radiobutton(mode_frame, text="Auto", variable=self.label_mode_var,
                        value="auto", command=self._toggle_label_mode).pack(side="left")
        ttk.Radiobutton(mode_frame, text="Fixed", variable=self.label_mode_var,
                        value="fixed", command=self._toggle_label_mode).pack(side="left", padx=(8,0))

        tk.Label(lbl_frame, text="Prefix:", bg=CARD, fg=FG, font=FONT).grid(
            row=2, column=0, sticky="w", pady=2)
        self.prefix_var = tk.StringVar(value="E20")
        self._entry(lbl_frame, self.prefix_var, width=8).grid(row=2, column=1,
                                                               sticky="w", padx=(6,0))

        # Auto-increment fields
        self.start_lbl = tk.Label(lbl_frame, text="Start #:", bg=CARD, fg=FG, font=FONT)
        self.start_lbl.grid(row=3, column=0, sticky="w", pady=2)
        self.start_var = tk.StringVar(value="1")
        self.start_entry = self._entry(lbl_frame, self.start_var, width=5)
        self.start_entry.grid(row=3, column=1, sticky="w", padx=(6,0))

        self.reset_var = tk.BooleanVar(value=False)
        self.reset_chk = ttk.Checkbutton(lbl_frame, text="Reset counter per room",
                                          variable=self.reset_var)
        self.reset_chk.grid(row=4, column=0, columnspan=2, sticky="w", pady=2)

        tk.Label(lbl_frame, text="Text height:", bg=CARD, fg=FG, font=FONT).grid(
            row=5, column=0, sticky="w", pady=2)
        self.texth_var = tk.StringVar(value="250")
        self._entry(lbl_frame, self.texth_var, width=8).grid(row=5, column=1,
                                                              sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Y Offset:", bg=CARD, fg=FG, font=FONT).grid(
            row=6, column=0, sticky="w", pady=2)
        self.offset_var = tk.StringVar(value="700")
        self._entry(lbl_frame, self.offset_var, width=8).grid(row=6, column=1,
                                                               sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="X Offset:", bg=CARD, fg=FG, font=FONT).grid(
            row=7, column=0, sticky="w", pady=2)
        self.xoffset_var = tk.StringVar(value="0")
        self._entry(lbl_frame, self.xoffset_var, width=8).grid(row=7, column=1,
                                                                sticky="w", padx=(6,0))

        # ── Place button ────────────────────
        btn_frame = tk.Frame(outer, bg=BG, pady=8)
        btn_frame.pack(fill="x")
        self._btn(btn_frame, "⚡  Place Lights", self._run, ACC,
                  font=("Segoe UI", 11, "bold"), pady=10).pack(fill="x")

        # ── Log ─────────────────────────────
        log_frame = ttk.LabelFrame(outer, text="  Log", padding=6)
        log_frame.pack(fill="both", expand=True, pady=(8,0))
        self.log = scrolledtext.ScrolledText(
            log_frame, height=8, bg="#0f0f1a", fg="#a5f3fc",
            font=("Consolas", 9), borderwidth=0, relief="flat",
            insertbackground="white"
        )
        self.log.pack(fill="both", expand=True)
        self.log.configure(state="disabled")

    def _btn(self, parent, text, cmd, bg="#3f3f5a", font=("Segoe UI", 10), pady=6):
        return tk.Button(parent, text=text, command=cmd,
                         bg=bg, fg="white", font=font,
                         relief="flat", cursor="hand2",
                         activebackground="#5b5b7a",
                         activeforeground="white",
                         padx=12, pady=pady, bd=0)

    def _entry(self, parent, var, width=12):
        return tk.Entry(parent, textvariable=var, width=width,
                        bg="#313145", fg="#e2e8f0", font=("Segoe UI", 10),
                        relief="flat", insertbackground="white",
                        highlightthickness=1, highlightcolor="#7c3aed")

    # ── HELPERS ─────────────────────────────

    def _log(self, msg, color=None):
        self.log.configure(state="normal")
        self.log.insert("end", msg + "\n")
        self.log.see("end")
        self.log.configure(state="disabled")

    def _set_status(self, msg, ok=True):
        self.status_var.set(msg)
        self.status_dot.configure(fg="#22c55e" if ok else "#ef4444")

    def _update_bulge_label(self, *_):
        b = self.bulge_var.get()
        if abs(b) < 0.01:
            txt = "straight line"
        else:
            deg = math.degrees(4 * math.atan(abs(b)))
            txt = f"≈ {deg:.0f}° arc"
        self.bulge_lbl.configure(text=txt)

    def _toggle_wire(self):
        state = "normal" if self.wire_var.get() else "disabled"
        self.bulge_scale.configure(state=state)

    def _toggle_labels(self):
        pass  # fields stay visible; logic skips if unchecked

    def _toggle_label_mode(self):
        """Show/hide auto-increment fields based on selected mode."""
        is_auto = self.label_mode_var.get() == "auto"
        state = "normal" if is_auto else "disabled"
        self.start_entry.configure(state=state)
        self.reset_chk.configure(state=state)
        self.start_lbl.configure(fg="#e2e8f0" if is_auto else "#555570")

    def _select_all(self):
        self.rect_listbox.select_set(0, "end")

    # ── AUTOCAD CONNECTION ───────────────────

    def _connect_autocad(self):
        """Launch one background thread that owns ALL COM work: connect + scan."""
        threading.Thread(target=self._com_worker, daemon=True).start()

    def _com_worker(self):
        """
        Single thread that owns every COM call.
        Rule: COM objects are NEVER passed to other threads.
        All AutoCAD work (connect, scan, place) happens here.
        """
        pythoncom.CoInitialize()
        try:
            acad = win32com.client.Dispatch(
                win32com.client.GetActiveObject("AutoCAD.Application"))
            doc  = win32com.client.Dispatch(acad.ActiveDocument)
            ms   = win32com.client.Dispatch(doc.ModelSpace)
        except Exception as ex:
            msg = str(ex)
            self.after(0, lambda: self._set_status("Not connected — click Reconnect", ok=False))
            self.after(0, lambda: self._log(f"✗ Cannot connect: {msg}"))
            return

        # Store references — only used from this thread via the queue
        self._acad = acad
        self._doc  = doc
        self._ms   = ms

        name = doc.Name
        self.after(0, lambda: self._set_status(f"Connected: {name}", ok=True))
        self.after(0, lambda: self._log(f"✓ Connected to {name}"))

        # Immediately scan
        self._do_scan()

        # Event loop — wait for jobs from the UI thread
        while True:
            try:
                job = self._job_queue.get(timeout=0.2)
                if job is None:
                    break
                if job[0] == "scan":
                    self._do_scan()
                elif job[0] == "place":
                    self._do_place(*job[1:])
            except Exception:
                continue

    # ── SCAN ────────────────────────────────

    def _scan_rectangles(self):
        """Called from UI — posts a scan job to the COM thread via queue."""
        try:
            self._job_queue.put(("scan",))
        except Exception:
            self._log("✗ Not connected yet — click Reconnect")

    def _do_scan(self):
        """Runs on the COM thread."""
        self.rectangles = []
        try:
            ms    = win32com.client.Dispatch(self._doc.ModelSpace)
            count = ms.Count
            for i in range(count):
                try:
                    entity = win32com.client.Dispatch(ms.Item(i))
                    bounds = get_rectangle_bounds(entity)
                    if bounds:
                        # Store only the bounds (plain Python data, safe to share)
                        self.rectangles.append(bounds)
                except Exception:
                    continue

            rects = list(self.rectangles)
            def _update():
                self.rect_listbox.delete(0, "end")
                for i, (x0, y0, x1, y1) in enumerate(rects):
                    label = f"[{i+1}]  {x1-x0:.0f} × {y1-y0:.0f}  @ ({x0:.0f}, {y0:.0f})"
                    self.rect_listbox.insert("end", label)
                self._log(f"↺ Scanned: {len(rects)} rectangle(s) found")
            self.after(0, _update)
        except Exception as e:
            err = str(e)
            self.after(0, lambda: self._log(f"✗ Scan error: {err}"))

    # ── PLACE ────────────────────────────────

    def _run(self):
        if not hasattr(self, '_job_queue') or not hasattr(self, '_doc'):
            messagebox.showerror("Error", "Not connected to AutoCAD. Click Reconnect.")
            return

        sel = list(self.rect_listbox.curselection())
        if not sel:
            messagebox.showwarning("No selection", "Select at least one rectangle.")
            return

        fixture_label = self.fixture_var.get()
        block_name    = BLOCK_NAMES.get(fixture_label)
        if not block_name:
            messagebox.showerror("Error", "Invalid fixture type."); return

        try:
            n_lights = int(self.count_var.get())
            assert n_lights > 0
        except Exception:
            messagebox.showerror("Error", "Light count must be a positive integer."); return

        draw_wire   = self.wire_var.get()
        bulge       = self.bulge_var.get()
        draw_label  = self.label_var.get()
        prefix      = self.prefix_var.get().strip()
        label_mode  = self.label_mode_var.get()   # "auto" or "fixed"
        reset_per   = self.reset_var.get()

        try:
            label_start = int(self.start_var.get())
        except Exception:
            label_start = 1

        try:
            text_h   = float(self.texth_var.get())
            text_off = float(self.offset_var.get())
            text_xoff= float(self.xoffset_var.get())
        except Exception:
            text_h, text_off, text_xoff = 250.0, 700.0, 0.0

        # Post placement job to the COM thread
        try:
            self._job_queue.put(("place", sel, block_name, n_lights, draw_wire, bulge,
                                 draw_label, prefix, label_start, reset_per, text_h, text_off, text_xoff, label_mode))
        except Exception as e:
            messagebox.showerror("Error", f"Could not queue job: {e}")

    def _do_place(self, sel, block_name, n_lights, draw_wire, bulge,
                  draw_label, prefix, label_start, reset_per,
                  text_h, text_off, text_xoff=0.0, label_mode="auto"):
        # Already on the COM thread — no CoInitialize needed
        doc = self._doc
        ms  = win32com.client.Dispatch(self._doc.ModelSpace)
        self.after(0, lambda: self._log("─" * 48))

        # Verify block exists
        try:
            self._doc.Blocks.Item(block_name)
        except Exception:
            self.after(0, lambda: self._log(
                f"✗ Block '{block_name}' not found. Insert it manually first."))
            return

        light_layer = LAYER_MAP.get(block_name, "ECLAIRAGE")
        ensure_layer(self._doc, light_layer)
        ensure_layer(self._doc, WIRE_LAYER,  color=6)
        ensure_layer(self._doc, LABEL_LAYER, color=7)

        total       = 0
        all_refs    = []
        label_index = label_start

        for idx in sel:
            x0, y0, x1, y1 = self.rectangles[idx]
            pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

            if reset_per:
                label_index = label_start

            # Blocks
            self._doc.ActiveLayer = self._doc.Layers.Item(light_layer)
            for (px, py) in pts:
                ref = ms.InsertBlock(make_point(px, py, 0.0), block_name,
                                     BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
                all_refs.append(ref)
                total += 1

            # Wire
            if draw_wire and len(pts) >= 2:
                self._doc.ActiveLayer = self._doc.Layers.Item(WIRE_LAYER)
                try:
                    wire = draw_bulge_wire(ms, pts, bulge)
                    if wire:
                        all_refs.append(wire)
                except Exception as we:
                    werr = str(we)
                    self.after(0, lambda: self._log(f"    ✗ Wire error: {werr}"))

            # Labels
            if draw_label:
                self._doc.ActiveLayer = self._doc.Layers.Item(LABEL_LAYER)
                for (px, py) in pts:
                    if label_mode == "fixed":
                        txt = prefix          # exact fixed label, no number
                    else:
                        txt = f"{prefix}.{label_index}"
                        label_index += 1
                    lbl = add_label(ms, px, py, txt, text_h, text_off, text_xoff)
                    if lbl:
                        all_refs.append(lbl)

            msg = (f"✓ Room [{idx+1}]: {len(pts)} lights  {cols}×{rows} grid"
                   f"  wire={'arc' if draw_wire else 'no'}"
                   f"  labels={'yes' if draw_label else 'no'}")
            self.after(0, lambda m=msg: self._log(m))

        self._doc.ActiveLayer = self._doc.Layers.Item("0")
        self._acad.ZoomExtents()
        self._doc.Regen(True)
        self._doc.Save()

        summary = f"✓ Done — {total} light(s) placed and saved."
        self.after(0, lambda: self._log(summary))
        self.after(0, lambda: self._set_status(summary, ok=True))
        self.after(0, lambda: messagebox.showinfo("Done", summary))


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    app = LightingApp()
    app.mainloop()